# Sprint 9 · Webinar 29 · Data Analytics práctico  
**Taller:** Manejo y lectura de errores + Debugging en Python  
**Duración:** 100 minutos · **Nivel:** Principiante

> En esta sesión práctica aprenderás a **leer errores**, **entender por qué ocurren** y **corregirlos** con un método paso a paso.  
> El objetivo no es memorizar excepciones, sino dominar un **proceso de diagnóstico** que puedas aplicar a cualquier proyecto de Data Analytics.


## Fecha
> Completa aquí la fecha de la sesión.


## Objetivo de la sesión práctica (100 min)

Al finalizar esta clase, podrás:

1. Leer un **traceback** y extraer lo importante: **tipo de error**, **mensaje**, **línea** y **causa probable**.
2. Reconocer errores comunes en Data Analytics con Python:  
   `NameError`, `TypeError`, `ValueError`, `KeyError`, `IndexError`, `AttributeError`, `FileNotFoundError`, `ZeroDivisionError`.
3. Aplicar un flujo simple de debugging:
   - Reproducir el error
   - Aislar el problema (la mínima parte que falla)
   - Inspeccionar variables (`type`, `len`, `head`, `unique`, etc.)
   - Corregir y validar con pruebas rápidas
4. Usar técnicas básicas para **prevenir errores**: validaciones, `assert`, y `try/except` con mensajes claros.


## Agenda sugerida (100 minutos)

1. Contexto del taller + checklist de habilidades (5 min)  
2. **Actividad 0 (breakout rooms):** leer un traceback como detective (10 min)  
3. Ejercicio 1: anatomía del error y el call stack (15 min)  
4. Ejercicio 2: `TypeError` en operaciones con pandas (15 min)  
5. Ejercicio 3: `KeyError` y limpieza de nombres de columnas (15 min)  
6. Ejercicio 4: `ValueError` al convertir fechas / números (15 min)  
7. Ejercicio 5: `IndexError` + `ZeroDivisionError` (15 min)  
8. Ejercicio 6: `try/except` + prevención (10 min)  
9. Mini-reto final + cierre (5 min)


## Contexto del taller

En Data Analytics es normal que tu código falle por:

- **Errores de código**: variables mal escritas, funciones usadas mal, índices fuera de rango.
- **Errores por datos**: columnas con espacios, números en texto, fechas inválidas, valores faltantes.
- **Errores por entorno**: rutas de archivos, librerías no instaladas.

La habilidad clave es **leer el mensaje**, **ubicar la línea** y **entender qué esperaba Python vs qué recibió**.

📌 Regla de oro: **la última línea del traceback suele ser la más importante** (tipo de error + mensaje).


## Dataset para la sesión

Usaremos un dataset **sintético** de ventas (pequeño) para practicar errores típicos con pandas.

Columnas:
- `order_id` — id del pedido  
- `date` — fecha del pedido (texto; incluye algunos errores intencionales)  
- `region` — región  
- `units` — unidades (texto, no número: causa errores)  
- `price` — precio unitario (número)  
- `channel` — canal (web / store)  

También guardaremos el dataset en un CSV local para practicar `FileNotFoundError`.


---

# Actividad 0 · Calentamiento (10 min)

### Instrucciones (breakout rooms, 3–4 personas)

Lee cada traceback (abajo) y responde:

1) **Tipo de error** (ej. `KeyError`)  
2) **Qué significa en palabras simples**  
3) **Qué revisarías primero** para arreglarlo

### Traceback A
```text
NameError: name 'df_sales' is not defined
```

### Traceback B
```text
KeyError: 'revenue'
```

### Traceback C
```text
TypeError: can't multiply sequence by non-int of type 'float'
```

### Pista
- Revisa la **última línea** del traceback.
- Pregunta: “¿Qué esperaba Python?” vs “¿Qué recibió?”

### Puesta en común (2 min)
Cada equipo comparte 1 traceback y su diagnóstico.


In [ ]:
# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


### Crear dataset sintético (para la práctica)

> Ejecuta esta celda para crear `df` y un archivo local `ventas_sinteticas.csv`.


In [ ]:
# ============================================================
# Dataset sintético de ventas (pequeño, con problemas intencionales)
# ============================================================
n = 40

df = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "date": pd.date_range("2026-01-10", periods=n, freq="D").astype(str),
    "region": np.random.choice(["Norte", "Sur", "Centro"], size=n, p=[0.4, 0.3, 0.3]),
    "units": np.random.choice(["1", "2", "3", "4", "two", ""], size=n, p=[0.25, 0.25, 0.2, 0.15, 0.1, 0.05]),
    "price": np.round(np.random.uniform(10, 80, size=n), 2),
    "channel": np.random.choice(["web", "store"], size=n, p=[0.6, 0.4]),
})

# Fechas inválidas intencionales (para provocar ValueError en algunos intentos)
df.loc[5, "date"] = "2026-13-40"   # fecha imposible
df.loc[17, "date"] = "not_a_date" # texto inválido

# Espacios en el nombre de una columna (para provocar KeyError al acceder)
df.rename(columns={"region": "region "}, inplace=True)

df.head()


In [ ]:
# Guardar a CSV local (para practicar FileNotFoundError)
df.to_csv("ventas_sinteticas.csv", index=False)

print("Archivo guardado:", "ventas_sinteticas.csv")
print("Dimensiones:", df.shape)


---

# Ejercicio 1 · Anatomía de un error (traceback) (15 min)

## Fundamentación teórica

Un **traceback** es el historial de llamadas (call stack) que muestra **qué funciones se ejecutaron** hasta que ocurrió el error.

Cómo leerlo (método práctico):
1. Ve a la **última línea**: tipo de error + mensaje.
2. Sube a la línea donde ocurre (en notebook suele ser la celda actual).
3. Identifica: **qué variable/objeto** está involucrado y **qué operación** se intentó.

## Descripción

Vas a ejecutar un bloque que falla (intencional). Tu trabajo es:
- Identificar el tipo de error
- Corregirlo con el cambio mínimo


### Tu turno: ejecuta este código (tiene un error intencional)

In [ ]:
# Código con error intencional
# Objetivo: ver el error, leerlo, y arreglarlo.

print("Filas del dataset:", df_sales.shape[0])  # <- df_sales no existe


### Pista
- Busca el nombre de la variable que realmente creamos (mira celdas anteriores).


In [ ]:
# ✅ Solución (cambio mínimo)
print("Filas del dataset:", df.shape[0])


### Extra: ver el call stack con funciones

En proyectos reales el error puede aparecer dentro de una función.
Ejecuta esto para ver cómo el traceback te guía hacia la línea exacta.


In [ ]:
def calcular_ticket_promedio(dataframe):
    # Error intencional: columna mal escrita (tiene un espacio al final)
    return dataframe["region"].value_counts()

calcular_ticket_promedio(df)


### Pista
- Revisa `df.columns` para ver el nombre exacto de la columna.


In [ ]:
# ✅ Solución
print("Columnas:", list(df.columns))

# La columna se llama 'region ' (con espacio al final)
df["region "].value_counts().head()


---

# Ejercicio 2 · `TypeError` en operaciones con pandas (15 min)

## Fundamentación teórica

`TypeError` ocurre cuando intentas una operación entre **tipos incompatibles**.

En Data Analytics pasa mucho por:
- números guardados como texto (`"2"`, `"3"`, `"two"`, `""`)
- fechas como texto
- mezclar `str` con `float`

✅ Herramientas útiles:
- `df.dtypes`
- `df["col"].unique()`
- `pd.to_numeric(..., errors="coerce")`

## Descripción

Queremos crear una columna `revenue = units * price`.

El código falla porque `units` es texto.


### Tu turno: ejecuta el código (tiene error intencional)

In [ ]:
# Código con error intencional
df["revenue"] = df["units"] * df["price"]
df[["units", "price", "revenue"]].head()


### Pista
1) Mira los valores de `units` con `unique()`  
2) Convierte a número con `pd.to_numeric(..., errors="coerce")`  
3) Decide qué hacer con valores inválidos (`NaN`): ¿eliminarlos? ¿reemplazarlos?

In [ ]:
# ✅ Solución paso a paso

# 1) Inspección
print("Tipo de units:", df["units"].dtype)
print("Valores únicos de units:", df["units"].unique())

# 2) Convertir a número (valores inválidos -> NaN)
df["units_num"] = pd.to_numeric(df["units"], errors="coerce")

# 3) Revisar cuántos quedaron inválidos
print("Unidades inválidas (NaN):", df["units_num"].isna().sum())

# 4) Reemplazar inválidos por 0 (una opción simple para la sesión)
df["units_num"] = df["units_num"].fillna(0)

# 5) Calcular revenue
df["revenue"] = df["units_num"] * df["price"]

df[["units", "units_num", "price", "revenue"]].head(10)


✅ Validación rápida: ¿hay revenue negativo? ¿tiene sentido el rango?

In [ ]:
df["revenue"].describe()


---

# Ejercicio 3 · `KeyError` y nombres de columnas (15 min)

## Fundamentación teórica

`KeyError` en pandas significa: **“esa columna (o clave) no existe”**.

Causas comunes:
- nombre mal escrito (`"Region"` vs `"region"`)
- espacios invisibles (`"region "` vs `"region"`)
- cambios de nombre sin actualizar el resto del código

✅ Herramientas útiles:
- `df.columns`
- `df.columns.str.strip()`
- `df.rename(...)`

## Descripción

Vamos a calcular revenue total por región con `groupby`.

El código falla por un `KeyError`.


### Tu turno: ejecuta el código (tiene error intencional)

In [ ]:
# Código con error intencional
# (1) agrupamos por region
# (2) sumamos revenue

ventas_region = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
ventas_region


### Pista
- Imprime `df.columns` y busca si `region` existe EXACTAMENTE así.
- Solución típica: limpiar espacios en nombres de columnas con `.str.strip()`.


In [ ]:
# ✅ Solución

print("Columnas originales:", list(df.columns))

# 1) Limpiar espacios a los nombres
df.columns = df.columns.str.strip()

print("Columnas limpiadas:", list(df.columns))

# 2) Ahora sí funciona
ventas_region = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
ventas_region


✅ Bonus: una práctica común es estandarizar a minúsculas (`lower()`) para evitar errores.


In [ ]:
# Bonus (opcional): estandarizar nombres
df.columns = df.columns.str.strip().str.lower()
df.head(2)


---

# Ejercicio 4 · `ValueError` al convertir fechas / números (15 min)

## Fundamentación teórica

`ValueError` ocurre cuando el **tipo es correcto** (por ejemplo, estás intentando convertir a fecha),
pero el **valor es inválido**:

- `"2026-13-40"` no es una fecha válida
- `"not_a_date"` tampoco
- `"N/A"` no se puede convertir a número

✅ Estrategia:
- Convierte con `errors="coerce"` para transformar inválidos en `NaT` (fechas) o `NaN` (números)
- Luego **cuenta** cuántos inválidos hay y decide qué hacer

## Descripción

Vamos a convertir `date` a datetime. Hay fechas inválidas, así que el intento directo falla.


### Tu turno: ejecuta el código (tiene error intencional)

In [ ]:
# Código con error intencional
df["date_dt"] = pd.to_datetime(df["date"])
df[["date", "date_dt"]].head()


### Pista
- Usa `errors="coerce"` en `pd.to_datetime`.
- Luego revisa cuántos `NaT` aparecen y en qué filas.


In [ ]:
# ✅ Solución

df["date_dt"] = pd.to_datetime(df["date"], errors="coerce")

# ¿Cuántas fechas inválidas?
invalid_dates = df["date_dt"].isna().sum()
print("Fechas inválidas (NaT):", invalid_dates)

# Ver las filas problemáticas
df.loc[df["date_dt"].isna(), ["order_id", "date", "date_dt"]]


✅ Validación rápida: rango de fechas y ejemplo de extracción de mes/semana.


In [ ]:
print("Mín:", df["date_dt"].min())
print("Máx:", df["date_dt"].max())

# Ejemplo de features de fecha (con NaT, pandas maneja bien)
df["month"] = df["date_dt"].dt.month
df[["date", "date_dt", "month"]].head(8)


---

# Ejercicio 5 · `IndexError` + `ZeroDivisionError` (15 min)

## Fundamentación teórica

### `IndexError`
Sucede cuando intentas acceder a una posición que **no existe**:
- `lista[10]` cuando la lista tiene 3 elementos

### `ZeroDivisionError`
Sucede cuando divides por cero, típicamente por:
- filtros que dejan un dataframe vacío
- conteos que dan 0

✅ Patrón de prevención:
- valida longitudes (`len(...)`)
- valida si el dataframe está vacío (`df.empty`)
- usa condicionales simples antes de dividir

## Descripción
Vamos a construir dos errores típicos y luego los arreglamos.


### 5.1 `IndexError`: acceder a un elemento que no existe

In [ ]:
# Código con error intencional
top_canales = df["channel"].value_counts().index.tolist()
print("Canales:", top_canales)

# Queremos el "tercer" canal (posición 2), pero ¿existe?
print("Tercer canal:", top_canales[2])


### Pista
- Imprime `len(top_canales)` antes de acceder.
- Si no existe, elige otra estrategia (por ejemplo, tomar el primero o usar `.get()` en series).


In [ ]:
# ✅ Solución

top_canales = df["channel"].value_counts()
print(top_canales)

if len(top_canales) >= 3:
    print("Tercer canal:", top_canales.index[2])
else:
    print("No hay 3 canales. Canales disponibles:", list(top_canales.index))


### 5.2 `ZeroDivisionError`: división por cero por filtro vacío

In [ ]:
# Código con error intencional
# Filtramos una región que NO existe para provocar un dataframe vacío
df_marte = df[df["region"] == "Marte"]

tasa_web = (df_marte[df_marte["channel"] == "web"].shape[0]) / df_marte.shape[0]
print("Tasa web en Marte:", tasa_web)


### Pista
- Revisa `df_marte.shape` y `df_marte.empty`.
- Si está vacío, tu código debería responder con un mensaje claro (y no dividir).


In [ ]:
# ✅ Solución

df_marte = df[df["region"] == "Marte"]
print("Filas en df_marte:", df_marte.shape[0])

if df_marte.empty:
    print("No hay datos para la región 'Marte'. Revisa el filtro o los valores únicos.")
else:
    tasa_web = (df_marte[df_marte["channel"] == "web"].shape[0]) / df_marte.shape[0]
    print("Tasa web en Marte:", tasa_web)


---

# Ejercicio 6 · `try/except` + prevención (10 min)

## Fundamentación teórica

`try/except` es útil cuando:
- el error **puede pasar** por una causa externa (archivo no existe, dato inválido)
- quieres dar un **mensaje claro** y continuar con el programa

Buenas prácticas:
- Captura **excepciones específicas** (no uses `except:` a ciegas)
- Incluye mensajes de acción: “revisa ruta”, “revisa columnas”, etc.
- Usa `assert` para validar supuestos en tus datos

## Descripción

1) Provocaremos `FileNotFoundError` al leer un CSV con ruta incorrecta.  
2) Lo manejaremos con `try/except`.  
3) Agregaremos una validación con `assert`.


### Tu turno: ejecuta el código (tiene error intencional)

In [ ]:
# Código con error intencional (ruta incorrecta)
df_archivo = pd.read_csv("ventas_sinteticas_NO_EXISTE.csv")
df_archivo.head()


### Pista
- Usa `try/except FileNotFoundError`.
- Luego lee el archivo correcto: `ventas_sinteticas.csv`.


In [ ]:
# ✅ Solución: manejo específico + mensaje claro

try:
    df_archivo = pd.read_csv("ventas_sinteticas_NO_EXISTE.csv")
except FileNotFoundError:
    print("No encontré el archivo. Revisa el nombre/ruta. Intentaré con el archivo correcto...")
    df_archivo = pd.read_csv("ventas_sinteticas.csv")

df_archivo.head()


### Bonus: validaciones simples para prevenir errores

Ejemplo: asegurarnos de que `price` sea no-negativo y que exista la columna `date`.


In [ ]:
# Bonus: assert (fallará si el supuesto no se cumple)
assert "date" in df_archivo.columns, "Falta la columna 'date'. Revisa el CSV."
assert (df_archivo["price"] >= 0).all(), "Hay precios negativos. Revisa el origen de datos."

print("Validaciones básicas: OK")


---

# Mini-reto final · Debuggea un script de EDA (5 min + 5 min)

## Descripción
Debajo hay un “mini-script” con varios errores típicos.  
Tu objetivo es hacerlo correr y producir una tabla final con:

- ventas totales por `region`
- ventas totales por `channel`

### Instrucciones
1) Ejecuta la celda (fallará).  
2) Arregla un error a la vez (método: **leer última línea del traceback**).  
3) No reescribas todo: haz cambios mínimos.

### Pista general (método)
- Revisa `df.columns`  
- Revisa tipos (`dtypes`)  
- Convierte datos (`to_numeric`, `to_datetime`)  
- Valida antes de agrupar (`assert`)


In [ ]:
# Mini-script con errores intencionales (para que lo depures)

df_debug = df.copy()

# 1) Error por nombre de columna (ojo: aquí usamos un nombre incorrecto a propósito)
#    (en muchos datasets reales pasa por espacios invisibles o mayúsculas/minúsculas)
df_debug["region "] = df_debug["region "].astype(str).str.upper()

# 2) Error de tipos: units es texto
df_debug["revenue"] = df_debug["units"] * df_debug["price"]

# 3) Error de fechas
df_debug["date_dt"] = pd.to_datetime(df_debug["date"])

# 4) Reporte final
reporte_region = df_debug.groupby("region")["revenue"].sum()
reporte_channel = df_debug.groupby("channel")["revenue"].sum()

print("=== Ventas por región ===")
print(reporte_region.sort_values(ascending=False))

print("\n=== Ventas por canal ===")
print(reporte_channel.sort_values(ascending=False))


### ✅ Solución completa (referencia)

> Úsala solo después de intentar el reto.


In [ ]:
# ✅ Solución completa del mini-reto (referencia)

df_debug = df.copy()

# 1) Limpiar nombres de columnas
df_debug.columns = df_debug.columns.str.strip()

# 2) Asegurar que region existe y es texto
df_debug["region"] = df_debug["region"].astype(str).str.upper()

# 3) units a numérico + revenue
df_debug["units_num"] = pd.to_numeric(df_debug["units"], errors="coerce").fillna(0)
df_debug["revenue"] = df_debug["units_num"] * df_debug["price"]

# 4) date a datetime (con coerción)
df_debug["date_dt"] = pd.to_datetime(df_debug["date"], errors="coerce")

# 5) Reporte final
reporte_region = df_debug.groupby("region")["revenue"].sum().sort_values(ascending=False)
reporte_channel = df_debug.groupby("channel")["revenue"].sum().sort_values(ascending=False)

print("=== Ventas por región ===")
print(reporte_region)

print("\n=== Ventas por canal ===")
print(reporte_channel)


---

# Cierre · Reporte final (5 min)

En 3–5 líneas, escribe:

1) ¿Cuál fue el error más común hoy y por qué apareció?  
2) ¿Qué checklist usarás la próxima vez que tu código falle?  
3) ¿Qué validación preventiva (`assert` o chequeo) agregarías a tus proyectos?

### Takeaways (para recordar)
- Lee el traceback **de abajo hacia arriba** (la última línea manda).
- Reproduce el error y **haz el cambio mínimo**.
- Inspecciona antes de suponer (`columns`, `dtypes`, `head`, `unique`).
- En Data Analytics: muchos “errores de código” nacen de **datos mal tipados o mal nombrados**.
